# 🤖 GenAI Course — Module 3: MCP Protocol Deep Dive

**Model Context Protocol (MCP)** is an open standard that enables AI models to securely interact with external tools, data sources, and services. In this module, we'll explore the protocol internals: how messages are structured, how sessions are managed, and how errors are handled.

---

### 📚 Topics Covered
1. JSON-RPC 2.0 — The Underlying Message Format
2. MCP Lifecycle — Initialization, Capability Negotiation, Session Management
3. Request / Response Structure
4. Notifications and Streaming
5. Error Handling and Status Codes
6. MCP Schema and Type System

---
> 💡 **Tip:** Run each cell in order. The demo cells use pure Python — no external MCP server needed!

---
## 1. JSON-RPC 2.0 — The Underlying Message Format

MCP is built on top of **JSON-RPC 2.0**, a lightweight remote procedure call protocol that uses JSON for encoding. Every MCP message — whether a request, response, or notification — conforms to this format.

### Key Rules of JSON-RPC 2.0
| Field | Description | Required? |
|-------|-------------|----------|
| `jsonrpc` | Always `"2.0"` | ✅ Yes |
| `method` | Name of the method to call | ✅ (requests) |
| `params` | Parameters for the method (object or array) | Optional |
| `id` | Unique request ID (integer or string). `null` for notifications | ✅ (requests/responses) |
| `result` | Successful response payload | ✅ (success responses) |
| `error` | Error object `{code, message, data}` | ✅ (error responses) |

In [1]:
import json

# ── Demo: Anatomy of a JSON-RPC 2.0 Request ──────────────────────────────────

rpc_request = {
    "jsonrpc": "2.0",         # Always exactly this string
    "id": 1,                  # Unique ID so the caller can match the response
    "method": "tools/call",   # The MCP method being invoked
    "params": {               # Method-specific parameters
        "name": "get_weather",
        "arguments": {"city": "Bangalore", "unit": "celsius"}
    }
}

print("── JSON-RPC 2.0 Request ──")
print(json.dumps(rpc_request, indent=2))

── JSON-RPC 2.0 Request ──
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "tools/call",
  "params": {
    "name": "get_weather",
    "arguments": {
      "city": "Bangalore",
      "unit": "celsius"
    }
  }
}


In [3]:
# ── Demo: Successful JSON-RPC 2.0 Response ────────────────────────────────────

rpc_success_response = {
    "jsonrpc": "2.0",
    "id": 1,                  # Matches the request id
    "result": {               # Present ONLY on success
        "content": [
            {"type": "text", "text": "Current temperature in Bangalore: 28°C, Partly Cloudy"}
        ],
        "isError": False
    }
}

print("── Successful Response ──")
print(json.dumps(rpc_success_response, indent=2))

# ── Demo: Error JSON-RPC 2.0 Response ────────────────────────────────────────
rpc_error_response = {
    "jsonrpc": "2.0",
    "id": 1,
    "error": {               # Present ONLY on failure — no 'result' key
        "code": -32602,
        "message": "Invalid params",
        "data": "'unit' must be 'celsius' or 'fahrenheit'"
    }
}

print("\n── Error Response ──")
print(json.dumps(rpc_error_response, indent=2))

── Successful Response ──
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "Current temperature in Bangalore: 28\u00b0C, Partly Cloudy"
      }
    ],
    "isError": false
  }
}

── Error Response ──
{
  "jsonrpc": "2.0",
  "id": 1,
  "error": {
    "code": -32602,
    "message": "Invalid params",
    "data": "'unit' must be 'celsius' or 'fahrenheit'"
  }
}


In [5]:
# ── Demo: Building a reusable JSON-RPC message factory ───────────────────────

import itertools

_id_counter = itertools.count(1)

def make_request(method: str, params: dict = None) -> dict:
    """Create a well-formed JSON-RPC 2.0 request."""
    msg = {"jsonrpc": "2.0", "id": next(_id_counter), "method": method}
    if params:
        msg["params"] = params
    return msg

def make_response(req_id, result=None, error=None) -> dict:
    """Create a well-formed JSON-RPC 2.0 response."""
    msg = {"jsonrpc": "2.0", "id": req_id}
    if error:
        msg["error"] = error
    else:
        msg["result"] = result
    return msg

# Try it out
req = make_request("resources/list", {"cursor": None})
print("Request:", json.dumps(req, indent=2))

resp = make_response(req["id"], result={"resources": [], "nextCursor": None})
print("\nResponse:", json.dumps(resp, indent=2))

Request: {
  "jsonrpc": "2.0",
  "id": 1,
  "method": "resources/list",
  "params": {
    "cursor": null
  }
}

Response: {
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "resources": [],
    "nextCursor": null
  }
}


---
## 2. MCP Lifecycle — Initialization, Capability Negotiation & Session Management

Every MCP session goes through a well-defined lifecycle:

```
Client                          Server
  │                               │
  │──── initialize ─────────────►│  Client announces its capabilities
  │◄─── InitializeResult ────────│  Server replies with its capabilities
  │──── notifications/initialized►│  Client signals it's ready
  │                               │
  │  ←── Active Session ──────►  │
  │                               │
  │──── (tool / resource calls) ─►│
  │◄─── (responses) ─────────────│
  │                               │
  │──── shutdown (optional) ─────►│
```

### Phases
| Phase | Description |
|-------|-------------|
| **Initialize** | Client sends protocol version + client capabilities |
| **Capability Negotiation** | Server responds with what it supports (tools, resources, prompts, logging, etc.) |
| **Initialized notification** | Client confirms it received the server's capabilities |
| **Active session** | Normal request/response exchange |
| **Shutdown** | Either side closes the transport |

In [7]:
# ── Demo: Simulating the MCP Initialization Handshake ───────────────────────

# Step 1 — Client sends 'initialize'
initialize_request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2024-11-05",   # MCP spec version
        "capabilities": {
            "roots": {"listChanged": True}, # Client can notify on root changes
            "sampling": {}                  # Client supports LLM sampling
        },
        "clientInfo": {
            "name": "MyGenAI-Client",
            "version": "1.0.0"
        }
    }
}

print("STEP 1 — Client → Server: initialize")
print(json.dumps(initialize_request, indent=2))

STEP 1 — Client → Server: initialize
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "initialize",
  "params": {
    "protocolVersion": "2024-11-05",
    "capabilities": {
      "roots": {
        "listChanged": true
      },
      "sampling": {}
    },
    "clientInfo": {
      "name": "MyGenAI-Client",
      "version": "1.0.0"
    }
  }
}


In [9]:
# Step 2 — Server responds with its capabilities
initialize_result = {
    "jsonrpc": "2.0",
    "id": 1,
    "result": {
        "protocolVersion": "2024-11-05",
        "capabilities": {
            "tools": {"listChanged": True},       # Server has tools, notifies on changes
            "resources": {
                "subscribe": True,                 # Supports resource subscriptions
                "listChanged": True
            },
            "prompts": {"listChanged": False},
            "logging": {}                          # Supports log emission
        },
        "serverInfo": {
            "name": "WeatherToolServer",
            "version": "2.3.1"
        }
    }
}

print("STEP 2 — Server → Client: InitializeResult")
print(json.dumps(initialize_result, indent=2))

STEP 2 — Server → Client: InitializeResult
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "protocolVersion": "2024-11-05",
    "capabilities": {
      "tools": {
        "listChanged": true
      },
      "resources": {
        "subscribe": true,
        "listChanged": true
      },
      "prompts": {
        "listChanged": false
      },
      "logging": {}
    },
    "serverInfo": {
      "name": "WeatherToolServer",
      "version": "2.3.1"
    }
  }
}


In [11]:
# Step 3 — Client sends the 'initialized' notification to confirm readiness
# Note: Notifications have NO 'id' field — they don't expect a response

initialized_notification = {
    "jsonrpc": "2.0",
    "method": "notifications/initialized"
    # ← no 'id' key!
}

print("STEP 3 — Client → Server: notifications/initialized")
print(json.dumps(initialized_notification, indent=2))
print("\n✅ Session is now active — both sides can exchange requests freely!")

STEP 3 — Client → Server: notifications/initialized
{
  "jsonrpc": "2.0",
  "method": "notifications/initialized"
}

✅ Session is now active — both sides can exchange requests freely!


In [13]:
# ── Demo: Simple MCP Session State Machine ────────────────────────────────────

from enum import Enum, auto

class SessionState(Enum):
    CREATED      = auto()
    INITIALIZING = auto()
    ACTIVE       = auto()
    CLOSING      = auto()
    CLOSED       = auto()

class MCPSession:
    def __init__(self):
        self.state = SessionState.CREATED
        self.server_capabilities = {}
        print(f"[Session] Created  →  state: {self.state.name}")

    def send_initialize(self):
        assert self.state == SessionState.CREATED, "Can only initialize a new session"
        self.state = SessionState.INITIALIZING
        print(f"[Client→] Sent 'initialize'  →  state: {self.state.name}")

    def receive_initialize_result(self, capabilities: dict):
        assert self.state == SessionState.INITIALIZING
        self.server_capabilities = capabilities
        print(f"[←Server] Got InitializeResult  |  capabilities: {list(capabilities.keys())}")

    def send_initialized(self):
        assert self.state == SessionState.INITIALIZING
        self.state = SessionState.ACTIVE
        print(f"[Client→] Sent 'notifications/initialized'  →  state: {self.state.name}")

    def close(self):
        self.state = SessionState.CLOSED
        print(f"[Session] Closed  →  state: {self.state.name}")

# Run the lifecycle
session = MCPSession()
session.send_initialize()
session.receive_initialize_result({"tools": True, "resources": True, "logging": True})
session.send_initialized()
print(f"\nServer supports tools? {'tools' in session.server_capabilities}")
session.close()

[Session] Created  →  state: CREATED
[Client→] Sent 'initialize'  →  state: INITIALIZING
[←Server] Got InitializeResult  |  capabilities: ['tools', 'resources', 'logging']
[Client→] Sent 'notifications/initialized'  →  state: ACTIVE

Server supports tools? True
[Session] Closed  →  state: CLOSED


---
## 3. Request / Response Structure

Once a session is active, the client can invoke three categories of MCP primitives:

| Primitive | Method prefix | Description |
|-----------|--------------|-------------|
| **Tools** | `tools/` | Functions the LLM can call (e.g., `tools/call`) |
| **Resources** | `resources/` | Read access to data (files, DB rows, APIs) |
| **Prompts** | `prompts/` | Pre-defined prompt templates |

Every request must have a unique `id`. The server's response carries the same `id` in either `result` (success) or `error` (failure).

In [15]:
# ── Demo: tools/list and tools/call ──────────────────────────────────────────

# 3a. List available tools
tools_list_request = {
    "jsonrpc": "2.0", "id": 10,
    "method": "tools/list",
    "params": {"cursor": None}   # Pagination cursor; None = first page
}

tools_list_response = {
    "jsonrpc": "2.0", "id": 10,
    "result": {
        "tools": [
            {
                "name": "get_weather",
                "description": "Returns current weather for a city.",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "city":  {"type": "string"},
                        "unit":  {"type": "string", "enum": ["celsius", "fahrenheit"]}
                    },
                    "required": ["city"]
                }
            },
            {
                "name": "send_email",
                "description": "Sends an email to a recipient.",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "to":      {"type": "string"},
                        "subject": {"type": "string"},
                        "body":    {"type": "string"}
                    },
                    "required": ["to", "subject", "body"]
                }
            }
        ],
        "nextCursor": None   # No more pages
    }
}

tools = tools_list_response["result"]["tools"]
print(f"Available tools ({len(tools)}):")
for t in tools:
    required = t["inputSchema"].get("required", [])
    print(f"  • {t['name']:15s} — {t['description']}  (required: {required})")

Available tools (2):
  • get_weather     — Returns current weather for a city.  (required: ['city'])
  • send_email      — Sends an email to a recipient.  (required: ['to', 'subject', 'body'])


In [17]:
# 3b. Call a tool
tool_call_request = {
    "jsonrpc": "2.0", "id": 11,
    "method": "tools/call",
    "params": {
        "name": "get_weather",
        "arguments": {"city": "Mumbai", "unit": "celsius"}
    }
}

# Simulated server response
tool_call_response = {
    "jsonrpc": "2.0", "id": 11,
    "result": {
        "content": [
            {"type": "text", "text": "Mumbai: 32°C, Humid, Chance of rain 60%"}
        ],
        "isError": False
    }
}

print("Tool Call Request:")
print(json.dumps(tool_call_request, indent=2))
print("\nTool Call Response:")
print(json.dumps(tool_call_response, indent=2))

# Extract result text
text_blocks = [c["text"] for c in tool_call_response["result"]["content"] if c["type"] == "text"]
print("\n📊 Extracted text output:", text_blocks)

Tool Call Request:
{
  "jsonrpc": "2.0",
  "id": 11,
  "method": "tools/call",
  "params": {
    "name": "get_weather",
    "arguments": {
      "city": "Mumbai",
      "unit": "celsius"
    }
  }
}

Tool Call Response:
{
  "jsonrpc": "2.0",
  "id": 11,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "Mumbai: 32\u00b0C, Humid, Chance of rain 60%"
      }
    ],
    "isError": false
  }
}

📊 Extracted text output: ['Mumbai: 32°C, Humid, Chance of rain 60%']


In [19]:
# 3c. Resource read request
resource_read_request = {
    "jsonrpc": "2.0", "id": 12,
    "method": "resources/read",
    "params": {
        "uri": "file:///data/students.csv"
    }
}

resource_read_response = {
    "jsonrpc": "2.0", "id": 12,
    "result": {
        "contents": [
            {
                "uri": "file:///data/students.csv",
                "mimeType": "text/csv",
                "text": "name,score\nAlice,92\nBob,87\nCarol,95"
            }
        ]
    }
}

csv_text = resource_read_response["result"]["contents"][0]["text"]
print("📁 Resource content (CSV):")
for line in csv_text.split("\n"):
    print(" ", line)

📁 Resource content (CSV):
  name,score
  Alice,92
  Bob,87
  Carol,95


---
## 4. Notifications and Streaming

### Notifications
Notifications are **fire-and-forget** messages with **no `id`** and **no response expected**.

Common MCP notifications:

| Direction | Method | Meaning |
|-----------|--------|---------|
| Client → Server | `notifications/initialized` | Session is ready |
| Client → Server | `notifications/roots/list_changed` | Client's root list changed |
| Server → Client | `notifications/tools/list_changed` | Tool list updated |
| Server → Client | `notifications/resources/updated` | A subscribed resource changed |
| Server → Client | `notifications/message` | Log message |
| Server → Client | `notifications/progress` | Progress update for long tasks |

### Streaming (Progress Tokens)
For long-running operations, the client sends a `progressToken` in `_meta`. The server then emits `notifications/progress` updates until the operation completes.

In [21]:
# ── Demo: Notification messages ───────────────────────────────────────────────

# Server notifies client that the tools list changed
tools_changed_notification = {
    "jsonrpc": "2.0",
    "method": "notifications/tools/list_changed"
    # ← no 'id' — notifications never have one
}

# Server sends a log message to the client
log_notification = {
    "jsonrpc": "2.0",
    "method": "notifications/message",
    "params": {
        "level": "info",
        "logger": "weather-tool",
        "data": "External weather API rate limit: 850/1000 requests used"
    }
}

for notif in [tools_changed_notification, log_notification]:
    has_id = 'id' in notif
    print(f"Method: {notif['method']}")
    print(f"  Has 'id': {has_id}  ← {'❌ wrong!' if has_id else '✅ correct — no id'}")
    print(json.dumps(notif, indent=2), "\n")

Method: notifications/tools/list_changed
  Has 'id': False  ← ✅ correct — no id
{
  "jsonrpc": "2.0",
  "method": "notifications/tools/list_changed"
} 

Method: notifications/message
  Has 'id': False  ← ✅ correct — no id
{
  "jsonrpc": "2.0",
  "method": "notifications/message",
  "params": {
    "level": "info",
    "logger": "weather-tool",
    "data": "External weather API rate limit: 850/1000 requests used"
  }
} 



In [23]:
# ── Demo: Streaming with progress tokens ─────────────────────────────────────

import time

# Client requests a long-running tool call with a progress token
long_task_request = {
    "jsonrpc": "2.0", "id": 20,
    "method": "tools/call",
    "params": {
        "name": "analyze_dataset",
        "arguments": {"dataset_id": "sales_2024"},
        "_meta": {
            "progressToken": "tok_abc123"   # Client supplies a unique token
        }
    }
}

print("Client sent long-running request with progressToken:", long_task_request["params"]["_meta"]["progressToken"])
print()

# Server emits progress notifications (simulated)
def simulate_progress_notifications(token: str, steps: list):
    total = len(steps)
    for i, step in enumerate(steps, 1):
        notification = {
            "jsonrpc": "2.0",
            "method": "notifications/progress",
            "params": {
                "progressToken": token,
                "progress": i,
                "total": total,
                "message": step
            }
        }
        pct = int(i / total * 100)
        bar = "█" * (pct // 10) + "░" * (10 - pct // 10)
        print(f"  [{bar}] {pct:3d}%  {step}")
        time.sleep(0.3)  # simulate async delay

steps = [
    "Loading dataset from storage",
    "Validating schema",
    "Running statistical analysis",
    "Generating visualizations",
    "Writing results"
]
simulate_progress_notifications("tok_abc123", steps)
print("\n✅ Final response delivered to id=20")

Client sent long-running request with progressToken: tok_abc123

  [██░░░░░░░░]  20%  Loading dataset from storage
  [████░░░░░░]  40%  Validating schema
  [██████░░░░]  60%  Running statistical analysis
  [████████░░]  80%  Generating visualizations
  [██████████] 100%  Writing results

✅ Final response delivered to id=20


---
## 5. Error Handling and Status Codes

MCP errors follow the JSON-RPC 2.0 error object schema:

```json
{ "code": -32600, "message": "Invalid Request", "data": "<optional extra info>" }
```

### Standard JSON-RPC Error Codes

| Code | Constant | When to use |
|------|----------|-------------|
| `-32700` | Parse Error | JSON could not be decoded |
| `-32600` | Invalid Request | `jsonrpc`, `method`, or `id` missing/wrong |
| `-32601` | Method Not Found | Unknown method name |
| `-32602` | Invalid Params | Params don't match schema |
| `-32603` | Internal Error | Server-side unexpected error |

### MCP-Specific Application Errors
Codes in the range **`-32000` to `-32099`** are reserved for implementation-defined errors.

| Code | Meaning |
|------|---------|
| `-32001` | Request Timeout |
| `-32002` | Resource Not Found |
| `-32003` | Tool Execution Failed (soft — `isError: true` in result is preferred) |

In [25]:
# ── Demo: Error code reference table ─────────────────────────────────────────

ERROR_CODES = {
    -32700: ("ParseError",       "Malformed JSON was received"),
    -32600: ("InvalidRequest",   "jsonrpc/id/method field is wrong or missing"),
    -32601: ("MethodNotFound",   "The requested method does not exist"),
    -32602: ("InvalidParams",    "Parameters are invalid or missing required fields"),
    -32603: ("InternalError",    "An unexpected error occurred on the server"),
    -32001: ("RequestTimeout",   "The operation took too long"),
    -32002: ("ResourceNotFound", "The requested URI does not exist"),
}

print(f"{'Code':>8}  {'Name':<20}  Description")
print("-" * 70)
for code, (name, desc) in ERROR_CODES.items():
    print(f"{code:>8}  {name:<20}  {desc}")

    Code  Name                  Description
----------------------------------------------------------------------
  -32700  ParseError            Malformed JSON was received
  -32600  InvalidRequest        jsonrpc/id/method field is wrong or missing
  -32601  MethodNotFound        The requested method does not exist
  -32602  InvalidParams         Parameters are invalid or missing required fields
  -32603  InternalError         An unexpected error occurred on the server
  -32001  RequestTimeout        The operation took too long
  -32002  ResourceNotFound      The requested URI does not exist


In [27]:
# ── Demo: Client-side error handling ─────────────────────────────────────────

class MCPError(Exception):
    def __init__(self, code: int, message: str, data=None):
        self.code = code
        self.message = message
        self.data = data
        super().__init__(f"[{code}] {message}")

def handle_rpc_response(response: dict):
    """Parse a JSON-RPC response, raising MCPError on failure."""
    if "error" in response:
        err = response["error"]
        raise MCPError(err["code"], err["message"], err.get("data"))
    return response["result"]

# Scenario 1 — success
success_resp = {"jsonrpc": "2.0", "id": 5, "result": {"temp": 29}}
try:
    result = handle_rpc_response(success_resp)
    print("✅ Success:", result)
except MCPError as e:
    print("❌ Error:", e)

# Scenario 2 — method not found
error_resp = {
    "jsonrpc": "2.0", "id": 6,
    "error": {"code": -32601, "message": "Method Not Found", "data": "tools/fly_rocket"}
}
try:
    result = handle_rpc_response(error_resp)
    print("✅ Success:", result)
except MCPError as e:
    print(f"❌ MCPError code={e.code} | {e.message} | extra: {e.data}")

✅ Success: {'temp': 29}
❌ MCPError code=-32601 | Method Not Found | extra: tools/fly_rocket


In [29]:
# ── Demo: Tool-level soft errors via isError flag ─────────────────────────────
# Even when the tool *executes*, it can signal a logical error via isError: True
# The LLM should then use this to inform the user, not retry blindly.

soft_error_response = {
    "jsonrpc": "2.0", "id": 7,
    "result": {
        "content": [
            {"type": "text", "text": "API quota exceeded for city: Tokyo. Try again in 60 seconds."}
        ],
        "isError": True    # ← soft error: RPC succeeded, but the tool action failed
    }
}

result = soft_error_response["result"]
if result.get("isError"):
    msg = result["content"][0]["text"]
    print(f"⚠️  Tool returned a soft error: {msg}")
else:
    print("✅ Tool result:", result["content"][0]["text"])

⚠️  Tool returned a soft error: API quota exceeded for city: Tokyo. Try again in 60 seconds.


---
## 6. MCP Schema and Type System

MCP uses **JSON Schema (Draft 7)** to define the shape of every tool's input. This allows:
- Clients to validate arguments **before** sending
- LLMs to understand what parameters to fill in
- Servers to reject malformed calls early

### Common JSON Schema Types in MCP

| Type | JSON Schema | Example |
|------|------------|--------|
| String | `{"type": "string"}` | `"Bangalore"` |
| Number | `{"type": "number"}` | `3.14` |
| Integer | `{"type": "integer"}` | `42` |
| Boolean | `{"type": "boolean"}` | `true` |
| Array | `{"type": "array", "items": {...}}` | `["a", "b"]` |
| Object | `{"type": "object", "properties": {...}}` | `{"k": "v"}` |
| Enum | `{"enum": ["a", "b"]}` | `"a"` |
| Null | `{"type": "null"}` | `null` |

In [31]:
# ── Demo: Defining a rich MCP tool schema ─────────────────────────────────────

search_flights_tool = {
    "name": "search_flights",
    "description": "Search for available flights between two airports on a given date.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "origin": {
                "type": "string",
                "description": "IATA airport code for departure (e.g. BLR)",
                "pattern": "^[A-Z]{3}$"
            },
            "destination": {
                "type": "string",
                "description": "IATA airport code for arrival (e.g. DEL)",
                "pattern": "^[A-Z]{3}$"
            },
            "date": {
                "type": "string",
                "description": "Departure date in ISO 8601 format",
                "format": "date"
            },
            "cabin_class": {
                "type": "string",
                "enum": ["economy", "premium_economy", "business", "first"],
                "default": "economy"
            },
            "max_results": {
                "type": "integer",
                "minimum": 1,
                "maximum": 50,
                "default": 10
            },
            "non_stop_only": {
                "type": "boolean",
                "default": False
            }
        },
        "required": ["origin", "destination", "date"],
        "additionalProperties": False   # Reject unknown fields
    }
}

schema = search_flights_tool["inputSchema"]
print(f"Tool: {search_flights_tool['name']}")
print(f"Required fields: {schema['required']}")
print(f"Optional fields: {[k for k in schema['properties'] if k not in schema['required']]}")
print(f"Allowed cabin classes: {schema['properties']['cabin_class']['enum']}")

Tool: search_flights
Required fields: ['origin', 'destination', 'date']
Optional fields: ['cabin_class', 'max_results', 'non_stop_only']
Allowed cabin classes: ['economy', 'premium_economy', 'business', 'first']


In [33]:
# ── Demo: Schema validation using jsonschema library ─────────────────────────

try:
    import jsonschema
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "jsonschema", "-q"])
    import jsonschema

def validate_tool_args(tool: dict, arguments: dict):
    """Validate tool arguments against its inputSchema."""
    try:
        jsonschema.validate(instance=arguments, schema=tool["inputSchema"])
        print(f"✅ Arguments are valid for '{tool['name']}'")
        return True
    except jsonschema.ValidationError as e:
        print(f"❌ Validation failed: {e.message}")
        return False

# Test 1: valid call
validate_tool_args(search_flights_tool, {
    "origin": "BLR",
    "destination": "DEL",
    "date": "2025-08-15",
    "cabin_class": "business"
})

# Test 2: missing required field
validate_tool_args(search_flights_tool, {
    "origin": "BLR",
    "cabin_class": "economy"   # missing 'destination' and 'date'
})

# Test 3: invalid enum value
validate_tool_args(search_flights_tool, {
    "origin": "BLR",
    "destination": "BOM",
    "date": "2025-09-01",
    "cabin_class": "ultra_luxury"   # not in enum
})

✅ Arguments are valid for 'search_flights'
❌ Validation failed: 'destination' is a required property
❌ Validation failed: 'ultra_luxury' is not one of ['economy', 'premium_economy', 'business', 'first']


False

In [35]:
# ── Demo: MCP Content Types ────────────────────────────────────────────────────
# Tool results can contain multiple content blocks of different types.

multi_content_result = {
    "content": [
        {
            "type": "text",
            "text": "Found 3 flights from BLR to DEL on 2025-08-15."
        },
        {
            "type": "text",
            "text": "Cheapest option: IndiGo 6E-234 | Dep 06:15 | Arr 08:45 | ₹4,820"
        },
        {
            "type": "image",
            "data": "<base64-encoded PNG of a seat map>",
            "mimeType": "image/png"
        },
        {
            "type": "resource",
            "resource": {
                "uri": "flights://BLR-DEL/2025-08-15/results.json",
                "mimeType": "application/json",
                "text": "{\"flights\": [...]}"
            }
        }
    ],
    "isError": False
}

content_type_map = {
    "text": "Plain text (shown to user / LLM)",
    "image": "Base64 image (visual feedback)",
    "resource": "Embedded resource reference"
}

print("Content blocks in result:")
for i, block in enumerate(multi_content_result["content"]):
    t = block["type"]
    print(f"  [{i}] type={t:10s} → {content_type_map.get(t, 'unknown')}")

Content blocks in result:
  [0] type=text       → Plain text (shown to user / LLM)
  [1] type=text       → Plain text (shown to user / LLM)
  [2] type=image      → Base64 image (visual feedback)
  [3] type=resource   → Embedded resource reference


---
## 🔄 End-to-End Mini Simulation

Let's put it all together: a minimal in-process MCP server and client communicating via Python queues.

In [39]:
# ── Demo: Minimal in-process MCP simulation ───────────────────────────────────

import queue, threading, time, json, random

# ── Simulated MCP Server ──────────────────────────────────────────────────────
class TinyMCPServer:
    TOOLS = [
        {"name": "add",    "description": "Add two numbers",    "inputSchema": {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a","b"]}},
        {"name": "greet",  "description": "Greet a person",     "inputSchema": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]}},
    ]

    def handle(self, req: dict) -> dict | None:
        method = req.get("method")
        req_id = req.get("id")         # None for notifications
        params = req.get("params", {})

        if method == "initialize":
            return {"jsonrpc":"2.0","id":req_id,"result":{"protocolVersion":"2024-11-05","capabilities":{"tools":{}},"serverInfo":{"name":"TinyServer","version":"0.1"}}}

        if method == "notifications/initialized":
            return None   # notifications never get a response

        if method == "tools/list":
            return {"jsonrpc":"2.0","id":req_id,"result":{"tools":self.TOOLS}}

        if method == "tools/call":
            name = params.get("name")
            args = params.get("arguments", {})
            if name == "add":
                result_text = str(args["a"] + args["b"])
            elif name == "greet":
                result_text = f"Hello, {args['name']}! Welcome to MCP."
            else:
                return {"jsonrpc":"2.0","id":req_id,"error":{"code":-32601,"message":"Method Not Found","data":name}}
            return {"jsonrpc":"2.0","id":req_id,"result":{"content":[{"type":"text","text":result_text}],"isError":False}}

        return {"jsonrpc":"2.0","id":req_id,"error":{"code":-32601,"message":"Method Not Found"}}


# ── Simulated MCP Client ──────────────────────────────────────────────────────
class TinyMCPClient:
    def __init__(self, server: TinyMCPServer):
        self.server = server
        self._id = 0
        self.ready = False

    def _next_id(self):
        self._id += 1
        return self._id

    def _send(self, method, params=None, is_notification=False):
        msg = {"jsonrpc":"2.0","method":method}
        if not is_notification:
            msg["id"] = self._next_id()
        if params:
            msg["params"] = params
        return self.server.handle(msg)

    def connect(self):
        print("[Client] Sending initialize...")
        resp = self._send("initialize", {"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"TinyClient","version":"0.1"}})
        info = resp["result"]["serverInfo"]
        print(f"[Client] Got InitializeResult from '{info['name']}' v{info['version']}")
        self._send("notifications/initialized", is_notification=True)
        print("[Client] Sent notifications/initialized — session ACTIVE\n")
        self.ready = True

    def list_tools(self):
        resp = self._send("tools/list")
        return resp["result"]["tools"]

    def call_tool(self, name, arguments):
        resp = self._send("tools/call", {"name":name,"arguments":arguments})
        if "error" in resp:
            raise MCPError(resp["error"]["code"], resp["error"]["message"])
        return resp["result"]


# ── Run the simulation ────────────────────────────────────────────────────────
server = TinyMCPServer()
client = TinyMCPClient(server)

client.connect()

tools = client.list_tools()
print(f"Available tools: {[t['name'] for t in tools]}\n")

r1 = client.call_tool("add", {"a": 42, "b": 58})
print(f"add(42, 58) → {r1['content'][0]['text']}")

r2 = client.call_tool("greet", {"name": "Priya"})
print(f"greet('Priya') → {r2['content'][0]['text']}")

try:
    client.call_tool("unknown_tool", {})
except MCPError as e:
    print(f"\n❌ Error caught: {e}")

[Client] Sending initialize...
[Client] Got InitializeResult from 'TinyServer' v0.1
[Client] Sent notifications/initialized — session ACTIVE

Available tools: ['add', 'greet']

add(42, 58) → 100
greet('Priya') → Hello, Priya! Welcome to MCP.

❌ Error caught: [-32601] Method Not Found
